# Relatório Final – Inflação, Política Monetária e Modelagem Preditiva

Este notebook (NB11) sintetiza todo o projeto em formato de **relatório analítico**:

- Diagnóstico da inflação (nível, composição, difusão, núcleos, momentum)
- Relação com a política monetária (SELIC, juros reais, defasagens, VAR)
- Decomposição macroeconômica (blocos)
- Curva de Phillips (atividade, hiato, inércia)
- Modelos de previsão (estatísticos e ML)

Objetivo: gerar **texto analítico semi-automático** + tabelas e gráficos de apoio.

Compatível com Jupytext: `11.relatorio_final.py`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

pd.set_option("display.float_format", "{:.4f}".format)
print("PYTHONPATH atualizado:", ROOT)

## 1. Carregar bases consolidadas

In [ ]:
path_selic = os.path.join(ROOT, "data", "processed", "ipca_selic_features.parquet")
path_blocos = os.path.join(ROOT, "data", "processed", "ipca_blocos_macro.parquet")
path_phillips = os.path.join(ROOT, "data", "processed", "curva_phillips_features.parquet")
path_eval_short = os.path.join(ROOT, "data", "processed", "eval_estatistico_ipca_t_plus_1.parquet")
path_eval_med = os.path.join(ROOT, "data", "processed", "eval_estatistico_ipca_12m_ahead.parquet")
path_ml_pred = os.path.join(ROOT, "data", "processed", "ml_predictions.parquet")

df_selic = pd.read_parquet(path_selic)
df_blocos = pd.read_parquet(path_blocos)
df_ph = pd.read_parquet(path_phillips)
df_eval_short = pd.read_parquet(path_eval_short)
df_eval_med = pd.read_parquet(path_eval_med)
df_ml = pd.read_parquet(path_ml_pred)

for df_ in [df_selic, df_blocos, df_ph, df_eval_short, df_eval_med, df_ml]:
    df_.index = pd.to_datetime(df_.index)

df_selic.tail(), df_blocos.tail(), df_ph.tail()

## 2. Funções auxiliares para texto automático

In [ ]:
def sinal(x: float) -> str:
    if x > 0:
        return "aceleração"
    elif x < 0:
        return "desaceleração"
    return "estabilidade"


def direcao(x: float) -> str:
    if x > 0:
        return "acima"
    elif x < 0:
        return "abaixo"
    return "em linha com"


def format_pct(x: float, casas: int = 1) -> str:
    return f"{x:.{casas}f}".replace(".", ",")

## 3. Diagnóstico recente da inflação

In [ ]:
# Últimos dados de IPCA e núcleo
ultimo_ipca = df_selic["ipca"].iloc[-1]
penultimo_ipca = df_selic["ipca"].iloc[-2]

ipca_12m_ultimo = df_selic["ipca_12m"].iloc[-1]
ipca_12m_penultimo = df_selic["ipca_12m"].iloc[-2]

if "nucleo_aparado" in df_ph.columns:
    nucleo_ultimo = df_ph["nucleo_aparado"].iloc[-1]
    nucleo_penultimo = df_ph["nucleo_aparado"].iloc[-2]
else:
    nucleo_ultimo = np.nan
    nucleo_penultimo = np.nan

texto_diagnostico = []

texto_diagnostico.append(
    f"No último dado disponível, o IPCA mensal foi de {format_pct(ultimo_ipca)}%, "
    f"{sinal(ultimo_ipca - penultimo_ipca)} em relação ao mês anterior "
    f"({format_pct(penultimo_ipca)}%)."
)

texto_diagnostico.append(
    f"A inflação acumulada em 12 meses está em {format_pct(ipca_12m_ultimo)}%, "
    f"{sinal(ipca_12m_ultimo - ipca_12m_penultimo)} na margem."
)

if not np.isnan(nucleo_ultimo):
    texto_diagnostico.append(
        f"O núcleo de inflação (média aparada) registrou {format_pct(nucleo_ultimo)}% no mês, "
        f"contra {format_pct(nucleo_penultimo)}% no mês anterior, sugerindo "
        f"{sinal(nucleo_ultimo - nucleo_penultimo)} da tendência subjacente."
    )

print("\n".join(texto_diagnostico))

## 4. Composição da inflação por blocos

In [ ]:
cols_blocos_12m = [c for c in df_blocos.columns if c.startswith("bloco_12m_")]
df_blocos_12m = df_blocos[cols_blocos_12m].iloc[-1].sort_values(ascending=False)

texto_blocos = []
texto_blocos.append("Na decomposição em 12 meses, os blocos que mais contribuíram para a inflação foram:")

for nome, valor in df_blocos_12m.items():
    if abs(valor) < 0.05:
        continue
    texto_blocos.append(f"- {nome.replace('bloco_12m_', '')}: {format_pct(valor)} p.p. em 12 meses")

print("\n".join(texto_blocos))

## 5. Política monetária e juros reais

In [ ]:
selic_ultima = df_selic["selic"].iloc[-1]
selic_real_ultima = df_selic["selic_real"].iloc[-1]

texto_politica = []

texto_politica.append(
    f"A taxa SELIC está em {format_pct(selic_ultima)}% ao ano, "
    f"enquanto os juros reais ex-post (SELIC - IPCA 12m) estão em "
    f"{format_pct(selic_real_ultima)}%."
)

if selic_real_ultima > 0:
    texto_politica.append(
        "Isso indica postura monetária contracionista, em linha com o objetivo de "
        "convergir a inflação para a meta."
    )
else:
    texto_politica.append(
        "Os juros reais se encontram em terreno pouco contracionista ou até estimulativo, "
        "o que tende a sustentar a demanda."
    )

print("\n".join(texto_politica))

## 6. Curva de Phillips – hiato e inércia

In [ ]:
texto_phillips = []

if "hiato_proxy" in df_ph.columns:
    hiato_ultimo = df_ph["hiato_proxy"].iloc[-1]
    texto_phillips.append(
        f"O hiato proxy (diferença entre serviços e bens industriais em 12 meses) "
        f"está em {format_pct(hiato_ultimo)} p.p., sugerindo pressão "
        f"{'de demanda' if hiato_ultimo > 0 else 'desinflacionária'} sobre a inflação de serviços."
    )

if "nucleo_lag_1" in df_ph.columns:
    texto_phillips.append(
        "A presença de inércia inflacionária é capturada pelo núcleo defasado, "
        "que se mostrou estatisticamente relevante na Curva de Phillips estimada."
    )

print("\n".join(texto_phillips))

## 7. Desempenho dos modelos de previsão

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

# curto prazo
df_short = df_eval_short.join(df_ml[["y_test_short", "y_pred_short"]], how="inner")

rmse_arima, mae_arima = metrics(df_short["ipca_t_test"], df_short["arima_ipca_t_plus_1"])
rmse_arimax, mae_arimax = metrics(df_short["ipca_t_test"], df_short["arimax_ipca_t_plus_1"])
rmse_ml_short, mae_ml_short = metrics(df_short["ipca_t_test"], df_short["y_pred_short"])

# médio prazo
df_med = df_eval_med.join(df_ml[["y_test_med", "y_pred_med"]], how="inner")

rmse_reg, mae_reg = metrics(df_med["ipca_12m_t_plus_12_test"], df_med["reg_ipca_12m_t_plus_12"])
rmse_ml_med, mae_ml_med = metrics(df_med["ipca_12m_t_plus_12_test"], df_med["y_pred_med"])

texto_modelos = []

texto_modelos.append(
    f"Para a previsão de IPCA mensal (t+1), o RMSE dos modelos foi aproximadamente:\n"
    f"- ARIMA: {rmse_arima:.3f}\n"
    f"- ARIMAX: {rmse_arimax:.3f}\n"
    f"- ML (melhor modelo): {rmse_ml_short:.3f}"
)

texto_modelos.append(
    f"Para a previsão de IPCA 12 meses à frente, o RMSE foi aproximadamente:\n"
    f"- Regressão estatística: {rmse_reg:.3f}\n"
    f"- ML (melhor modelo): {rmse_ml_med:.3f}"
)

print("\n\n".join(texto_modelos))

## 8. Geração de um resumo executivo consolidado

In [ ]:
resumo = []

resumo.append("### 1. Inflação recente")
resumo.extend(texto_diagnostico)
resumo.append("")

resumo.append("### 2. Composição da inflação")
resumo.extend(texto_blocos)
resumo.append("")

resumo.append("### 3. Política monetária")
resumo.extend(texto_politica)
resumo.append("")

resumo.append("### 4. Curva de Phillips")
resumo.extend(texto_phillips)
resumo.append("")

resumo.append("### 5. Modelos de previsão")
resumo.extend(texto_modelos)

relatorio_markdown = "\n".join(resumo)
print(relatorio_markdown)

## 9. Exportar relatório em Markdown

In [ ]:
output_dir = os.path.join(ROOT, "reports")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "relatorio_final_ipca.md")

with open(output_path, "w", encoding="utf-8") as f:
    f.write(relatorio_markdown)

print("Relatório salvo em:", output_path)

## 10. Observação final

Este relatório é um **ponto de partida**:
- Você pode editar o `.md` gerado, ajustar a linguagem, incluir gráficos exportados dos notebooks.
- Pode também gerar versões em PDF/HTML a partir desse markdown.

A partir daqui, o projeto está completo:
- diagnóstico
- decomposição
- política monetária
- Phillips
- modelos estatísticos
- modelos de ML
- dashboard
- relatório final.